# 08 — Low-Resource Learning Study
## Hybrid Quantum-Classical NLP Framework

Evaluates the robustness of the **final optimized** Hybrid Quantum-Classical model under varying amounts of training data.

The architecture is **fixed** — only the training set size changes.

**⚠️ Run Cell 1 → Restart Runtime → Run Cell 2 onwards**

In [ ]:
# ============================================================
# CELL 1 — Installation  (then Runtime → Restart Runtime)
# ============================================================
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    return r

print("Step 1/6: Wiping stale packages from disk...")
run("pip uninstall -y pennylane pennylane-lightning autoray jax jaxlib 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'pennylane*' -exec rm -rf {} + 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'PennyLane*' -exec rm -rf {} + 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'autoray*'   -exec rm -rf {} + 2>/dev/null || true")
print("  ✓ Stale files wiped")

print("Step 2/6: Installing numpy + scipy (ABI-compatible pair)...")
run("pip install -q --no-cache-dir numpy==1.26.4")
run("pip install -q --no-cache-dir scipy==1.13.1")
print("  ✓ numpy/scipy OK")

print("Step 3/6: Pinning JAX to a pennylane-compatible version...")
run("pip install -q --no-cache-dir 'jax==0.4.26' 'jaxlib==0.4.26'")
print("  ✓ JAX pinned")

print("Step 4/6: Installing autoray + pennylane (matched pair)...")
run("pip install -q --no-cache-dir autoray==0.6.12")
run("pip install -q --no-cache-dir pennylane==0.38.0 pennylane-lightning==0.38.0")
print("  ✓ pennylane/autoray OK")

print("Step 5/6: Installing NLP/ML stack...")
run("pip install -q --no-cache-dir "
    "numba>=0.57.1 "
    "transformers>=4.40.0 "
    "datasets>=2.19.0 "
    "scikit-learn>=1.4.0 "
    "gensim>=4.3.0 "
    "nltk>=3.8.0 "
    "seaborn>=0.13.0 "
    "tabulate>=0.9.0 "
    "pyyaml>=6.0 "
    "pandas==2.2.2 "
    "requests==2.32.4 "
    "'rich>=12.4.4,<14' "
    "'cachetools>=4.2.1,<7' "
    "'pillow<12.0' "
    "'tomlkit>=0.12.0,<0.14.0' "
    "openpyxl>=3.1.0"
)
print("  ✓ NLP/ML stack OK")

print("Step 6/6: Verifying installed versions...")
for pkg in ["pennylane", "autoray", "numpy", "scipy", "jax"]:
    run(f"pip show {pkg} | grep -E 'Name|Version'")

print()
print("=" * 60)
print("✅ Done! NOW: Runtime → Restart Runtime")
print("   Then run Cell 2 onwards.")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 2 — Verify Imports  (AFTER restarting runtime)
# ============================================================
import importlib

print("Verifying all imports...")
print("-" * 50)

ok = True
checks = [
    ("numpy",       "1.26"),
    ("scipy",       "1.13"),
    ("jax",         "0.4.26"),
    ("autoray",     "0.6"),
    ("pennylane",   "0.38"),
    ("torch",       None),
    ("transformers",None),
    ("sklearn",     None),
]

for mod, prefix in checks:
    try:
        m   = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        chk = prefix is None or ver.startswith(prefix)
        print(f"  {'✅' if chk else '⚠️ '} {mod:<15} {ver}")
        if not chk: ok = False
    except Exception as e:
        print(f"  ❌ {mod:<15} {e}")
        ok = False

print()
try:
    import pennylane as qml
    dev = qml.device("default.qubit", wires=2)
    @qml.qnode(dev)
    def _t():
        qml.Hadamard(wires=0)
        qml.CNOT(wires=[0,1])
        return qml.expval(qml.PauliZ(0))
    print(f"  ✅ pennylane circuit  result={float(_t()):.4f}")
except Exception as e:
    print(f"  ❌ pennylane circuit  {e}")
    ok = False

print()
if ok:
    print("✅ All checks passed — safe to continue!")
else:
    print("❌ Checks failed.")
    print("   → Runtime → Factory Reset Runtime, then rerun Cell 1.")


In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive
# ============================================================
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/HybridQuantumNLP"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"✅ Drive mounted. Project dir: {DRIVE_PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 4A — Load Project: Upload ZIP  (run this OR 4B)
# ============================================================
from google.colab import files
import zipfile, glob, os, sys

print("Upload your project zip...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
PROJECT_DIR = "/content/project"

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(PROJECT_DIR)

run_files = glob.glob(f"{PROJECT_DIR}/**/run.py", recursive=True)
if run_files:
    PROJECT_DIR = os.path.dirname(run_files[0])

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"✅ Project loaded: {PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 4B — Load Project: From Drive  (run this OR 4A)
# ============================================================
import os, sys, shutil, zipfile

DRIVE_ZIP_PATH = "/content/drive/MyDrive/HybridQuantumNLP/JU_project.zip"
PROJECT_DIR    = "/content/project"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
    z.extractall(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"✅ Project loaded from Drive: {PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 5 — Configuration
# ============================================================
# ⚠️  ONLY modify LANGUAGE, TRAINING_FRACTIONS, and MAX_SAMPLES.
# Everything else is the FINAL optimized pipeline from Notebooks 04–07.
# ============================================================
import torch

# ============= USER CONFIGURATION =============
LANGUAGE = "bengali"        # "english", "hindi", or "bengali"

TRAINING_FRACTIONS = [
    1.00,
    0.50,
    0.25,
    0.10,
    0.05,
]

MAX_SAMPLES = None           # None = full dataset; e.g. 500 for quick test
# ===============================================

# --- Final optimized pipeline (from Notebooks 04–07) ---
BEST_EMBEDDING   = "indicbert"    # From 04_embedding_study
BEST_VQC         = "v1"           # From 05_vqc_ablation_study
BEST_ACTIVATION  = "sigmoid"      # From 06_activation_ablation
BEST_OPTIMIZER   = "RMSProp"      # From 07_optimizer_ablation
N_QUBITS         = 4
N_LAYERS         = 2
LEARNING_RATE    = 0.01
QUANTUM_EPOCHS   = 30
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
SEED             = 42

print("=" * 60)
print("  LOW-RESOURCE LEARNING STUDY")
print("=" * 60)
print(f"  Language:       {LANGUAGE}")
print(f"  Fractions:      {TRAINING_FRACTIONS}")
print(f"  Max samples:    {MAX_SAMPLES or 'Full dataset'}")
print(f"  Embedding:      {BEST_EMBEDDING}")
print(f"  VQC:            Custom VQC-{BEST_VQC}")
print(f"  Activation:     {BEST_ACTIVATION}")
print(f"  Optimizer:      {BEST_OPTIMIZER}")
print(f"  Qubits:         {N_QUBITS}")
print(f"  Layers:         {N_LAYERS}")
print(f"  Epochs:         {QUANTUM_EPOCHS}")
print(f"  LR:             {LEARNING_RATE}")
print(f"  Device:         {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"  Seed:           {SEED}")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 6 — Build Config Object
# ============================================================
from config import load_config

cfg = load_config(overrides={
    "device": DEVICE,
    "seed": SEED,
    "embeddings": {
        "method": BEST_EMBEDDING,
        "indicbert": {"model_name": "ai4bharat/IndicBERTv2-MLM-only"},
    },
    "quantum_template": {"n_qubits": N_QUBITS, "n_layers": N_LAYERS, "epochs": QUANTUM_EPOCHS},
    "custom_vqc": {
        "n_qubits": N_QUBITS,
        "n_layers": N_LAYERS,
        "epochs": QUANTUM_EPOCHS,
        "version": BEST_VQC,
        "learning_rate": LEARNING_RATE,
    },
})
print("✅ Config ready.")


In [ ]:
# ============================================================
# CELL 7 — Dataset Overview
# ============================================================
from data.registry import get_dataset_info, UNIFIED_LABELS

print("=" * 65)
for name, lang in [("go_emotions","English"),("emo_in_hindi","Hindi"),("emonoba","Bengali")]:
    info = get_dataset_info(name)
    src  = info.hf_id or info.github_url or info.path
    print(f"  {lang}: {info.name}")
    print(f"    Source: {info.source} ({src})")
    print(f"    Labels: {info.num_labels} → {info.label_names[:6]}")
print(f"  Unified labels: {UNIFIED_LABELS}")
print("=" * 65)


In [ ]:
# ============================================================
# CELL 8 — Dataset Load Test
# ============================================================
from data.loader import load_dataset_for_language

test_ds = load_dataset_for_language(LANGUAGE, cfg, max_samples=100)
print(test_ds.summary())
print(f"Labels: {test_ds.label_names}")
print(f"Sample: {test_ds.train['texts'][0][:120]}...")
print("✅ Dataset OK")


---
## Low-Resource Learning Study

**Fixed (Final Optimized Pipeline):**
- Embedding: BEST_EMBEDDING (from Notebook 04)
- VQC: Custom VQC-BEST_VQC (from Notebook 05)
- Activation: BEST_ACTIVATION (from Notebook 06)
- Optimizer: BEST_OPTIMIZER (from Notebook 07)

**Variable:**
- Training data fraction: 100%, 50%, 25%, 10%, 5%

**Evaluation:**
- Test set is **unchanged** across all fractions.
- Stratified sub-sampling maintains label distribution.

In [ ]:
# ============================================================
# CELL 10 — Training Utilities
# ============================================================
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from data.loader import load_dataset_for_language
from data.splitter import subsample
from preprocessing.pipeline import preprocess_dataset
from embeddings.factory import get_embedder, embed_dataset
from models.custom_vqc.hybrid_model import CustomVQCModel
from models.custom_vqc.feature_compressor import FeatureCompressor
from evaluation.metrics import compute_metrics, format_classification_report
from evaluation.metrics import compute_confusion_matrix
from visualization.plots import (
    print_results_table, plot_model_comparison,
    plot_training_curves, plot_confusion_matrix,
)

# --- Optimizer factory (reused from notebook 07) ---
def get_optimizer(name, params, lr):
    """Create optimizer by name."""
    name_lower = name.lower()
    if name_lower == "adam":
        return torch.optim.Adam(params, lr=lr)
    elif name_lower == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    elif name_lower == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    elif name_lower == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr)
    else:
        raise ValueError(f"Unknown optimizer: {name}")


def train_with_optimizer(model, optimizer_name, lr, X_train, y_train,
                          X_val, y_val, epochs, batch_size, device):
    """Train a CustomVQCModel with a specific optimizer.
    Reuses the same training logic as CustomVQCClassifier.train()
    but allows optimizer injection."""
    device = torch.device(device)
    model = model.to(device)

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    optimizer = get_optimizer(optimizer_name, model.parameters(), lr)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        # --- Training ---
        model.train()
        running_loss, running_acc, n_batches = 0.0, 0.0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False)
        for batch_x, batch_y in pbar:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            running_acc += (logits.argmax(dim=-1) == batch_y).float().mean().item()
            n_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        history["train_loss"].append(running_loss / max(n_batches, 1))
        history["train_acc"].append(running_acc / max(n_batches, 1))

        # --- Validation ---
        model.eval()
        val_loss, val_acc, val_batches = 0.0, 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                val_loss += loss.item()
                val_acc += (logits.argmax(dim=-1) == batch_y).float().mean().item()
                val_batches += 1

        history["val_loss"].append(val_loss / max(val_batches, 1))
        history["val_acc"].append(val_acc / max(val_batches, 1))

        print(f"  Epoch {epoch}/{epochs} — "
              f"train_loss={history['train_loss'][-1]:.4f}  "
              f"val_loss={history['val_loss'][-1]:.4f}  "
              f"val_acc={history['val_acc'][-1]:.4f}")

    history["training_time"] = time.perf_counter() - start_time
    return history


def predict_model(model, X, batch_size, device):
    """Generate predictions from a model."""
    model.eval()
    device_t = torch.device(device)
    tensor_x = torch.tensor(X, dtype=torch.float32).to(device_t)
    all_preds = []
    with torch.no_grad():
        loader = DataLoader(TensorDataset(tensor_x), batch_size=batch_size, shuffle=False)
        for (batch_x,) in loader:
            logits = model(batch_x)
            all_preds.append(logits.argmax(dim=-1).cpu().numpy())
    return np.concatenate(all_preds)


print("✅ Training utilities ready.")


In [ ]:
# ============================================================
# CELL 11 — Load Full Dataset & Embed ONCE
# ============================================================
import os
import random

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Loading and embedding full dataset...")
dataset = load_dataset_for_language(LANGUAGE, cfg, max_samples=MAX_SAMPLES)
dataset = preprocess_dataset(dataset, cfg)
embedder = get_embedder(cfg, LANGUAGE)
embeddings = embed_dataset(embedder, dataset)

X_train_full = embeddings["train"]
y_train_full = np.array(dataset.train["labels"])
texts_train_full = dataset.train["texts"]

X_val   = embeddings["val"]
y_val   = np.array(dataset.val["labels"])
X_test  = embeddings["test"]
y_test  = np.array(dataset.test["labels"])
texts_test = dataset.test["texts"]

input_dim   = X_train_full.shape[1]
num_labels  = dataset.num_labels
label_names = dataset.label_names
batch_size  = getattr(cfg.custom_vqc, "batch_size", 32)

print(f"  Full train: {len(y_train_full)}, Val: {len(y_val)}, Test: {len(y_test)}")
print(f"  Input dim: {input_dim}, Classes: {num_labels}")
print(f"  Labels: {label_names}")
print("✅ Data ready — test/val sets fixed across all fractions.")

# Patch activation if needed
ACTIVATION_MAP = {"tanh": "tanh", "sigmoid": "sigmoid", "relu": "relu", "identity": "none"}
act_internal = ACTIVATION_MAP.get(BEST_ACTIVATION, BEST_ACTIVATION)
if act_internal == "relu":
    FeatureCompressor._ACTIVATIONS["relu"] = nn.ReLU
    print("  Patched FeatureCompressor for relu")


In [ ]:
# ============================================================
# CELL 12 — Low-Resource Training Loop
# ============================================================
import os

for subdir in ["classification_reports", "confusion_matrices", "plots", "training_curves"]:
    os.makedirs(f"results/low_resource_learning/{subdir}", exist_ok=True)

lr_results = {}       # {fraction_str: {test_metrics, history, params, training_time, ...}}
lr_predictions = {}   # {fraction_str: {y_true, y_pred}}

for frac in TRAINING_FRACTIONS:
    frac_label = f"{int(frac*100)}%"
    print(f"\n{'='*65}")
    print(f"  TRAINING FRACTION: {frac_label} ({frac})")
    print(f"{'='*65}")

    try:
        # --- 1. Sub-sample training data (val/test unchanged) ---
        if frac >= 1.0:
            X_train_sub = X_train_full
            y_train_sub = y_train_full
        else:
            # Use subsample to get stratified indices
            idx_texts = [str(i) for i in range(len(y_train_full))]
            idx_labels = y_train_full.tolist()
            sub_texts, sub_labels = subsample(
                idx_texts, idx_labels,
                fraction=frac, stratified=True, seed=SEED
            )
            sub_indices = [int(t) for t in sub_texts]
            X_train_sub = X_train_full[sub_indices]
            y_train_sub = y_train_full[sub_indices]

        n_train = len(y_train_sub)
        print(f"  Training samples: {n_train} (from {len(y_train_full)})")
        print(f"  Val samples: {len(y_val)} (unchanged)")
        print(f"  Test samples: {len(y_test)} (unchanged)")

        # --- 2. Build fresh model ---
        model = CustomVQCModel(
            input_dim=input_dim,
            n_qubits=N_QUBITS,
            n_layers=N_LAYERS,
            num_classes=num_labels,
            version=BEST_VQC,
        )

        # Override activation if needed
        if act_internal != "tanh":
            model.compressor = FeatureCompressor(
                input_dim, N_QUBITS, activation=act_internal
            )

        model = model.to(DEVICE)
        params = sum(p.numel() for p in model.parameters())

        # --- 3. Train ---
        history = train_with_optimizer(
            model, BEST_OPTIMIZER, LEARNING_RATE,
            X_train_sub, y_train_sub, X_val, y_val,
            QUANTUM_EPOCHS, batch_size, DEVICE
        )

        training_time = history.get("training_time", 0)

        # --- 4. Evaluate on test set ---
        y_pred = predict_model(model, X_test, batch_size, DEVICE)
        test_metrics = compute_metrics(y_test, y_pred)

        # --- 5. Store results ---
        lr_results[frac_label] = {
            "fraction": frac,
            "n_train": n_train,
            "test_metrics": test_metrics,
            "history": history,
            "params": params,
            "training_time": training_time,
        }
        lr_predictions[frac_label] = {
            "y_true": y_test,
            "y_pred": y_pred,
        }

        # --- 6. Print ---
        print(f"\n  Results for {frac_label}:")
        print(f"  {'='*40}")
        print(f"  Accuracy:      {test_metrics['accuracy']:.4f}")
        print(f"  Precision:     {test_metrics['precision']:.4f}")
        print(f"  Recall:        {test_metrics['recall']:.4f}")
        print(f"  F1 Score:      {test_metrics['f1']:.4f}")
        print(f"  Training Time: {training_time:.1f}s")
        print(f"  Parameters:    {params:,}")

        # --- 7. Classification report ---
        report = format_classification_report(y_test, y_pred, label_names)
        print(f"\n  Classification Report ({frac_label}):")
        print(report)
        report_path = f"results/low_resource_learning/classification_reports/frac_{int(frac*100)}_report.txt"
        with open(report_path, "w") as f:
            f.write(f"Low-Resource Learning — {frac_label}\n")
            f.write(f"Language: {LANGUAGE}\n")
            f.write(f"Training samples: {n_train}\n\n")
            f.write(report)

    except Exception as e:
        print(f"  \u274c {frac_label} FAILED: {e}")
        import traceback; traceback.print_exc()
        lr_results[frac_label] = {"error": str(e), "fraction": frac}

print(f"\n{'='*65}")
print(f"  Low-resource learning loop complete!")
print(f"  Successful: {sum(1 for v in lr_results.values() if 'error' not in v)}/{len(TRAINING_FRACTIONS)}")
print(f"{'='*65}")


In [ ]:
# ============================================================
# CELL 13 — Summary Table
# ============================================================
import pandas as pd

rows = []
for frac_label, data in lr_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        m = data["test_metrics"]
        rows.append({
            "Training Fraction": frac_label,
            "Samples": data.get("n_train", "?"),
            "Accuracy":  round(m["accuracy"], 4),
            "Precision": round(m["precision"], 4),
            "Recall":    round(m["recall"], 4),
            "F1":        round(m["f1"], 4),
            "Training Time (s)": round(data.get("training_time", 0), 1),
            "Parameters": data.get("params", "?"),
        })
    else:
        rows.append({
            "Training Fraction": frac_label,
            "Samples": "—",
            "Accuracy": "FAILED",
            "Precision": "FAILED",
            "Recall": "FAILED",
            "F1": "FAILED",
            "Training Time (s)": "",
            "Parameters": "",
        })

results_df = pd.DataFrame(rows)

print("\n" + "=" * 90)
print("  LOW-RESOURCE LEARNING — SUMMARY TABLE")
print("=" * 90)
print(results_df.to_string(index=False))
print("=" * 90)

# Save CSV
results_df.to_csv("results/low_resource_learning/low_resource_results.csv", index=False)
print("\n✅ CSV saved to results/low_resource_learning/low_resource_results.csv")

# Save Excel
try:
    results_df.to_excel("results/low_resource_learning/low_resource_results.xlsx", index=False)
    print("✅ Excel saved to results/low_resource_learning/low_resource_results.xlsx")
except Exception as e:
    print(f"⚠️ Excel save failed: {e}")


In [ ]:
# ============================================================
# CELL 14 — Publication-Quality Metric vs Fraction Graphs
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("results/low_resource_learning/plots", exist_ok=True)

# Collect data for line plots
fractions_plot = []
acc_plot, prec_plot, rec_plot, f1_plot, time_plot = [], [], [], [], []

for frac_label, data in lr_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        fractions_plot.append(data["fraction"])
        m = data["test_metrics"]
        acc_plot.append(m["accuracy"])
        prec_plot.append(m["precision"])
        rec_plot.append(m["recall"])
        f1_plot.append(m["f1"])
        time_plot.append(data.get("training_time", 0))

# Sort by fraction (ascending)
if fractions_plot:
    sorted_idx = sorted(range(len(fractions_plot)), key=lambda i: fractions_plot[i])
    fractions_plot = [fractions_plot[i] for i in sorted_idx]
    acc_plot  = [acc_plot[i]  for i in sorted_idx]
    prec_plot = [prec_plot[i] for i in sorted_idx]
    rec_plot  = [rec_plot[i]  for i in sorted_idx]
    f1_plot   = [f1_plot[i]   for i in sorted_idx]
    time_plot = [time_plot[i] for i in sorted_idx]

    frac_labels = [f"{int(f*100)}%" for f in fractions_plot]

    # Style
    sns.set_style("whitegrid")
    palette = sns.color_palette("Set2", 5)

    def make_metric_plot(metric_values, metric_name, color, filename):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(frac_labels, metric_values, 'o-', color=color,
                linewidth=2.5, markersize=10, markeredgecolor='black', markeredgewidth=0.5)
        for i, (x, y) in enumerate(zip(frac_labels, metric_values)):
            ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points",
                        xytext=(0, 12), ha='center', fontsize=10, fontweight='bold')
        ax.set_xlabel("Training Fraction", fontsize=13)
        ax.set_ylabel(metric_name, fontsize=13)
        ax.set_title(f"Low-Resource Learning — {metric_name} vs Training Fraction ({LANGUAGE.title()})",
                      fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, max(metric_values) * 1.2 if metric_values else 1)
        plt.tight_layout()
        plt.savefig(f"results/low_resource_learning/plots/{filename}", dpi=300, bbox_inches="tight")
        plt.show()

    # 1. F1 vs Training Fraction
    make_metric_plot(f1_plot, "Weighted F1", palette[0], "f1_vs_fraction.png")

    # 2. Accuracy vs Training Fraction
    make_metric_plot(acc_plot, "Accuracy", palette[1], "accuracy_vs_fraction.png")

    # 3. Precision vs Training Fraction
    make_metric_plot(prec_plot, "Precision", palette[2], "precision_vs_fraction.png")

    # 4. Recall vs Training Fraction
    make_metric_plot(rec_plot, "Recall", palette[3], "recall_vs_fraction.png")

    # 5. Training Time vs Training Fraction
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(frac_labels, time_plot, color=palette[4], edgecolor="black", linewidth=0.5)
    for bar, t in zip(bars, time_plot):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{t:.1f}s", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_xlabel("Training Fraction", fontsize=13)
    ax.set_ylabel("Training Time (seconds)", fontsize=13)
    ax.set_title(f"Low-Resource Learning — Training Time vs Fraction ({LANGUAGE.title()})",
                  fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig("results/low_resource_learning/plots/training_time_vs_fraction.png", dpi=300, bbox_inches="tight")
    plt.show()

print("✅ All metric-vs-fraction plots saved to results/low_resource_learning/plots/")


In [ ]:
# ============================================================
# CELL 15 — Combined Metrics Overlay Plot
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

if fractions_plot:
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=(12, 7))

    metrics_data = [
        (f1_plot,   "Weighted F1", "o-"),
        (acc_plot,  "Accuracy",    "s--"),
        (prec_plot, "Precision",   "^:"),
        (rec_plot,  "Recall",      "D-."),
    ]
    colors = sns.color_palette("husl", len(metrics_data))

    for (vals, name, marker), color in zip(metrics_data, colors):
        ax.plot(frac_labels, vals, marker, color=color,
                linewidth=2.5, markersize=9, label=name, markeredgecolor='black', markeredgewidth=0.5)

    ax.set_xlabel("Training Fraction", fontsize=13)
    ax.set_ylabel("Score", fontsize=13)
    ax.set_title(f"Low-Resource Learning — All Metrics vs Training Fraction ({LANGUAGE.title()})",
                  fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig("results/low_resource_learning/plots/all_metrics_overlay.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("✅ Combined metrics overlay saved.")


In [ ]:
# ============================================================
# CELL 16 — Confusion Matrices
# ============================================================
from visualization.plots import plot_confusion_matrix

for frac_label, pred_data in lr_predictions.items():
    print(f"\nConfusion Matrix — {frac_label}")
    frac_int = frac_label.replace('%', '')
    plot_confusion_matrix(
        pred_data["y_true"],
        pred_data["y_pred"],
        label_names=label_names,
        title=f"Confusion Matrix — {frac_label} Training Data ({LANGUAGE.title()})",
        save_path=f"results/low_resource_learning/confusion_matrices/cm_frac_{frac_int}.png",
    )

print("\n✅ Confusion matrices saved to results/low_resource_learning/confusion_matrices/")


In [ ]:
# ============================================================
# CELL 17 — Training Curves per Fraction
# ============================================================
from visualization.plots import plot_training_curves

for frac_label, data in lr_results.items():
    if isinstance(data, dict) and "history" in data:
        h = data["history"]
        if "train_loss" in h and len(h["train_loss"]) > 1:
            frac_int = frac_label.replace('%', '')
            plot_training_curves(
                h,
                title=f"Training Curves — {frac_label} Data ({LANGUAGE.title()})",
                save_path=f"results/low_resource_learning/training_curves/curves_frac_{frac_int}.png"
            )

print("✅ Training curves saved.")


In [ ]:
# ============================================================
# CELL 18 — Best & Worst Analysis
# ============================================================
import json

# Find best fraction by F1
best_f1 = -1
best_acc = -1
best_frac = "100%"

for frac_label, data in lr_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        f1  = data["test_metrics"].get("f1", 0)
        acc = data["test_metrics"].get("accuracy", 0)
        if f1 > best_f1 or (f1 == best_f1 and acc > best_acc):
            best_f1  = f1
            best_acc = acc
            best_frac = frac_label

print(f"\n{'='*65}")
print(f"  LOW-RESOURCE LEARNING — ANALYSIS")
print(f"{'='*65}")
print(f"  Best F1:       {best_f1:.4f} (at {best_frac})")
print(f"  Best Accuracy: {best_acc:.4f} (at {best_frac})")

# Degradation analysis
if "100%" in lr_results and "test_metrics" in lr_results["100%"]:
    full_f1  = lr_results["100%"]["test_metrics"]["f1"]
    full_acc = lr_results["100%"]["test_metrics"]["accuracy"]
    print(f"\n  Degradation from 100% baseline:")
    print(f"  {'Fraction':<12} {'F1':<10} {'Δ F1':<10} {'Accuracy':<10} {'Δ Acc':<10}")
    print(f"  {'-'*52}")
    for frac_label, data in lr_results.items():
        if isinstance(data, dict) and "test_metrics" in data:
            m = data["test_metrics"]
            df1  = m["f1"]  - full_f1
            dacc = m["accuracy"] - full_acc
            print(f"  {frac_label:<12} {m['f1']:<10.4f} {df1:+<10.4f} {m['accuracy']:<10.4f} {dacc:+<10.4f}")

# Compute the "robustness" — what fraction of 100%-F1 is retained at 5%?
if "5%" in lr_results and "test_metrics" in lr_results.get("5%", {}):
    f1_5pct = lr_results["5%"]["test_metrics"]["f1"]
    retention = (f1_5pct / full_f1 * 100) if full_f1 > 0 else 0
    print(f"\n  F1 Retention at 5%: {retention:.1f}% of baseline")
    if retention > 80:
        print(f"  → Model shows STRONG robustness to data reduction.")
    elif retention > 60:
        print(f"  → Model shows MODERATE robustness to data reduction.")
    else:
        print(f"  → Model shows POOR robustness — performance degrades significantly.")

print(f"{'='*65}")

# Save summary JSON
results_summary = {
    "language": LANGUAGE,
    "best_f1": best_f1,
    "best_accuracy": best_acc,
    "best_fraction": best_frac,
    "pipeline": {
        "embedding": BEST_EMBEDDING,
        "vqc": f"Custom VQC-{BEST_VQC}",
        "activation": BEST_ACTIVATION,
        "optimizer": BEST_OPTIMIZER,
        "n_qubits": N_QUBITS,
        "n_layers": N_LAYERS,
    },
    "fractions": {
        k: {
            "n_train": v.get("n_train", "?"),
            "metrics": v.get("test_metrics", {}),
            "training_time": v.get("training_time", 0),
            "params": v.get("params", "?"),
        } if isinstance(v, dict) and "test_metrics" in v
        else {"error": v.get("error", "unknown")}
        for k, v in lr_results.items()
    },
}
with open("results/low_resource_learning/summary.json", "w") as f:
    json.dump(results_summary, f, indent=2, default=str)

print(f"\n✅ Summary saved to results/low_resource_learning/summary.json")


In [ ]:
# ============================================================
# CELL 19 — Publication-Ready Table (LaTeX-style)
# ============================================================
from IPython.display import display, Markdown

table_lines = [
    "## Low-Resource Learning Results\n",
    f"**Language:** {LANGUAGE.title()}\n",
    f"**Pipeline:** {BEST_EMBEDDING} → tanh → VQC-{BEST_VQC} → {BEST_OPTIMIZER}\n",
    "",
    "| Training Fraction | Samples | Accuracy | Precision | Recall | F1 | Time (s) |",
    "|:-:|:-:|:-:|:-:|:-:|:-:|:-:|",
]

for frac_label, data in lr_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        m = data["test_metrics"]
        table_lines.append(
            f"| {frac_label} | {data.get('n_train','?')} | "
            f"{m['accuracy']:.4f} | {m['precision']:.4f} | "
            f"{m['recall']:.4f} | {m['f1']:.4f} | "
            f"{data.get('training_time',0):.1f} |"
        )
    else:
        table_lines.append(f"| {frac_label} | — | FAILED | — | — | — | — |")

table_md = "\n".join(table_lines)
display(Markdown(table_md))

# Save markdown
with open("results/low_resource_learning/results_table.md", "w") as f:
    f.write(table_md)
print("\n✅ Markdown table saved to results/low_resource_learning/results_table.md")


In [ ]:
# ============================================================
# CELL 20 — Save All Results to Google Drive
# ============================================================
import os, shutil

results_src = os.path.join(PROJECT_DIR, "results", "low_resource_learning")
results_dst = os.path.join(DRIVE_PROJECT_DIR, "results", "low_resource_learning")

if os.path.exists(results_src):
    os.makedirs(results_dst, exist_ok=True)
    for root, dirs, files_list in os.walk(results_src):
        rel = os.path.relpath(root, results_src)
        dst = os.path.join(results_dst, rel)
        os.makedirs(dst, exist_ok=True)
        for f in files_list:
            shutil.copy2(os.path.join(root, f), os.path.join(dst, f))
    print(f"✅ Results saved to: {results_dst}")

    # List all saved files
    print("\nSaved files:")
    for root, dirs, flist in os.walk(results_dst):
        level = root.replace(results_dst, "").count(os.sep)
        indent = "  " * (level + 1)
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = "  " * (level + 2)
        for f in flist:
            print(f"{sub_indent}{f}")
else:
    print("⚠️  No results directory found — run experiments first.")
